# 🔬 Speech model comparison

Runs several speech models over the **same 5-minute clips from your own lectures** and writes them side by side to `Vorlesungen/Results` in your Drive. Your transcripts and the normal *Lecture Transcriber* are not touched.

**Before you start:** *Runtime → Change runtime type → **L4 GPU*** (the T4 can't run the bigger models properly). Use a fresh runtime, not one that just ran the Lecture Transcriber.

**Then:** *Runtime → Run all*. It takes about **40 minutes** (much of it downloading ~60 GB of models), roughly **1–2 compute units** on an L4. If a model fails, the reason is written into the key file in Drive. When it's done: *Runtime → Disconnect and delete runtime*.

In [ ]:
#@title ⚙️ Settings
DRIVE_FOLDER = "Vorlesungen"  #@param {type:"string"}
#@markdown **Models to compare**
whisper_batched = False  #@param {type:"boolean"}
whisper_batched_loud = True  #@param {type:"boolean"}
#@markdown ↑ the Lecture Transcriber's setup since 22.10. — the yardstick
whisper_sequential = False  #@param {type:"boolean"}
whisper_sequential_loud = False  #@param {type:"boolean"}
primeline_german = False  #@param {type:"boolean"}
primeline_turbo_de = False  #@param {type:"boolean"}
voxtral_mini = True  #@param {type:"boolean"}
voxtral_small_4bit = True  #@param {type:"boolean"}
qwen3_asr = True  #@param {type:"boolean"}
#@markdown ↑ *Voxtral Small* is the largest (downloads ~48 GB, needs L4 or A100)

# Clips: GoodNotes notebook (part of its name), recording (start of its id), minute, length
CLIPS = [
    {"id": "m13_hard", "notebook": "6_Inferenzstatistik", "recording": "8662089F",
     "start_min": 10, "minutes": 5, "why": "M13 Inferenzstatistik — distant lecturer, the weakest course"},
    {"id": "m13_quiet", "notebook": "4_Versuchsplanung", "recording": "FD34CADB",
     "start_min": 7, "minutes": 5, "why": "M13 Versuchsplanung — has quiet stretches (invented fillers?)"},
    {"id": "m5_clear", "notebook": "VL_DMM_02", "recording": "DA590AAD",
     "start_min": 10, "minutes": 5, "why": "M5 Itemanalyse — clear speaker"},
    {"id": "m1", "notebook": "M1_2_IH", "recording": "9B6153F3",
     "start_min": 20, "minutes": 5, "why": "M1 — clinical vocabulary"},
    {"id": "replikation_gap", "notebook": "3_Replikation", "recording": "74952E91",
     "start_min": 28, "minutes": 5, "why": "Replikation part 2, minute 28 — inside the 42 minutes without text"},
]

In [ ]:
#@title 1 · Setup
import os, subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ No GPU. Runtime → Change runtime type → L4 GPU → Save, then Run all again.")
print("✓ GPU:", gpu.stdout.strip())
if "T4" in gpu.stdout:
    print("⚠ T4: Voxtral Small is skipped and Voxtral/Qwen run slowly. L4 is recommended.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "faster-whisper", "pypdf", "wordfreq"],
               check=True)
if os.path.isdir("/content/whisper"):
    subprocess.run(["git", "-C", "/content/whisper", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/RKIein/whisper", "/content/whisper"], check=True)
sys.path.insert(0, "/content/whisper")
import importlib
for name in ("lecture_store", "colab_batch", "model_compare"):
    if name in sys.modules:                 # code updated since this session started → load the new version
        importlib.reload(sys.modules[name])
import model_compare
print("✓ Code ready")

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
RESULTS = f"{DRIVE}/{DRIVE_FOLDER}/Results"
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
#@title 2 · Cut the test clips from your GoodNotes backup
WORK = "/content/compare"
manifest = model_compare.prepare(DRIVE, CLIPS, WORK)

In [ ]:
#@title 3 · Run the models (the long part)
chosen = [name for name, on in [
    ("whisper-batched", whisper_batched), ("whisper-batched-loud", whisper_batched_loud),
    ("whisper-sequential", whisper_sequential), ("whisper-sequential-loud", whisper_sequential_loud),
    ("primeline-german", primeline_german), ("primeline-turbo-de", primeline_turbo_de),
    ("voxtral-mini", voxtral_mini), ("voxtral-small-4bit", voxtral_small_4bit),
    ("qwen3-asr", qwen3_asr)] if on]
results = model_compare.run_all(f"{WORK}/manifest.json", chosen, WORK)

In [ ]:
#@title 4 · Write the comparison to Drive
texts, key = model_compare.report(manifest, results, RESULTS)
print("✓ Saved to Drive →", DRIVE_FOLDER + "/Results:")
print("   ", os.path.basename(texts), "  (the versions, unnamed)")
print("   ", os.path.basename(key), "  (which version is which model + scores)")
print("Done. Runtime → Disconnect and delete runtime, so it stops using compute units.")